In [1]:
import os
import sys


# Set required root pointers
SHIFT_PATH = os.path.abspath(os.path.join(os.getcwd(), "../../..", "shift"))
if not os.path.isdir(SHIFT_PATH):
    raise FileNotFoundError(f"shift not found at {SHIFT_PATH}. Please clone or set correct path.")

PYPSA_EARTH_PATH = os.path.abspath(os.path.join(SHIFT_PATH,"..", "pypsa-earth"))
if not os.path.isdir(PYPSA_EARTH_PATH):
    raise FileNotFoundError(f"pypsa-earth not found at {PYPSA_EARTH_PATH}. Please clone or set correct path.")

# Change working directory to shift root
os.chdir(SHIFT_PATH)

sys.path.append(SHIFT_PATH)
sys.path.append(PYPSA_EARTH_PATH)

In [2]:
import logging
import xarray as xr
import geopandas as gpd
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
import pycountry

# Custom formatter to hide full paths in git commits
class RelativePathFormatter(logging.Formatter):
    def format(self, record):
        # Convert full path to relative path
        record.pathname = os.path.relpath(record.pathname)
        # Format: relative/path/file.py - message
        return f"{record.pathname} - {record.getMessage()}"

handler = logging.StreamHandler()
handler.setFormatter(RelativePathFormatter())
logger = logging.getLogger()
logger.handlers.clear()
logger.addHandler(handler)
logger.setLevel(logging.INFO)


In [3]:
tech_profiles_nc = {}
for technology in ["onwind", "offwind-ac", "solar"]:
    path = os.path.realpath(PYPSA_EARTH_PATH) + f"/resources/renewable_profiles/profile_{technology}.nc"
    ds = xr.open_dataset(path)
    tech_profiles_nc[technology] = ds
    logging.info(f"Loaded {technology} profiles")

c:\Users\JanLeopoldTautorus\Repos\shift\.pixi\envs\dev\Lib\site-packages\xarray\backends\plugins.py:109: RuntimeWarning: Engine 'cfgrib' loading failed:
Cannot find the ecCodes library
  external_backend_entrypoints = backends_dict_from_pkg(entrypoints_unique)
..\..\AppData\Local\Temp\ipykernel_10284\2480569425.py - Loaded onwind profiles
..\..\AppData\Local\Temp\ipykernel_10284\2480569425.py - Loaded offwind-ac profiles
..\..\AppData\Local\Temp\ipykernel_10284\2480569425.py - Loaded solar profiles


In [4]:
tech_profiles_nc["onwind"]

<xarray.Dataset> Size: 71MB
Dimensions:           (time: 8760, bus: 1009, y: 78, x: 131)
Coordinates:
  * time              (time) datetime64[ns] 70kB 2013-01-01 ... 2013-12-31T23...
  * bus               (bus) <U4 16kB '176' '180' '182' ... '1258' '1259' '1260'
  * y                 (y) float64 624B 32.4 32.7 33.0 33.3 ... 54.9 55.2 55.5
  * x                 (x) float64 1kB -31.5 -31.2 -30.9 -30.6 ... 6.9 7.2 7.5
Data variables:
    profile           (time, bus) float64 71MB ...
    weight            (bus) float64 8kB ...
    p_nom_max         (bus) float64 8kB ...
    potential         (y, x) float64 82kB ...
    average_distance  (bus) float64 8kB ...

In [5]:
onshore_regions_gpd = gpd.read_file(os.path.realpath(PYPSA_EARTH_PATH) + "/resources/bus_regions/regions_onshore.geojson")
offshore_regions_gpd = gpd.read_file(os.path.realpath(PYPSA_EARTH_PATH) + "/resources/bus_regions/regions_offshore.geojson")

# Convert ISO2 country codes to ISO3
onshore_regions_gpd['country'] = onshore_regions_gpd['country'].apply(lambda iso2: pycountry.countries.get(alpha_2=iso2).alpha_3)
offshore_regions_gpd['country'] = offshore_regions_gpd['country'].apply(lambda iso2: pycountry.countries.get(alpha_2=iso2).alpha_3)

# Rename offshore regions to avoid ID collisions (same numeric IDs exist in both datasets)
# Prefix offshore names with "OFF_" to make them globally unique
offshore_regions_gpd['name'] = 'OFF_' + offshore_regions_gpd['name'].astype(str)

# Compute area for each region (in km²)
# Reproject to Web Mercator (EPSG:3857) to get accurate area in meters, then convert to km²
onshore_regions_gpd['area_km2'] = onshore_regions_gpd.to_crs('EPSG:3857').geometry.area / 1e6
offshore_regions_gpd['area_km2'] = offshore_regions_gpd.to_crs('EPSG:3857').geometry.area / 1e6
logging.info(f"Onshore regions: {len(onshore_regions_gpd)}, mean area={onshore_regions_gpd['area_km2'].mean():.1f} km²")
logging.info(f"Offshore regions: {len(offshore_regions_gpd)}, mean area={offshore_regions_gpd['area_km2'].mean():.1f} km²")
logging.info(f"Deduplication: Offshore region names prefixed with 'OFF_' to avoid ID collisions")
logging.info(f"Conversion: Country codes converted from ISO2 to ISO3 for consistent downstream usage")

c:\Users\JanLeopoldTautorus\Repos\shift\.pixi\envs\dev\Lib\site-packages\pyogrio\core.py:34: RuntimeWarning: Could not detect GDAL data files. Set GDAL_DATA environment variable to the correct path.
  _init_gdal_data()
..\..\AppData\Local\Temp\ipykernel_10284\2768513724.py - Onshore regions: 1009, mean area=445.4 km²
..\..\AppData\Local\Temp\ipykernel_10284\2768513724.py - Offshore regions: 195, mean area=9487.9 km²
..\..\AppData\Local\Temp\ipykernel_10284\2768513724.py - Deduplication: Offshore region names prefixed with 'OFF_' to avoid ID collisions
..\..\AppData\Local\Temp\ipykernel_10284\2768513724.py - Conversion: Country codes converted from ISO2 to ISO3 for consistent downstream usage


In [6]:
def cluster_renewable_potentials(
    tech_profiles_nc,
    onshore_regions_gpd,
    offshore_regions_gpd,
    country_codes=None,
    total_clusters=10,
    onshore_ratio=0.8,
    random_state=42,
    separate_onshore_offshore=True
):
    
    """
    Cluster renewable resource potentials and capacity factors by geographic proximity.
    
    Aggregates total generation potential (from tech_profiles['potential']) and hourly 
    capacity factor time series across multiple regions into geographic clusters. The clustering 
    process averages CF timeseries, which distorts the resource profile. To preserve energy 
    conservation, we calculate p_nom_max = generation_potential / avg_cf_clustered. The 
    deviation of clustered p_nom_max from raw values is the clustering distortion metric.
    
    Parameters
    ----------
    tech_profiles_nc : dict[str, xr.Dataset]
        Technology profiles keyed by name (e.g., 'onwind', 'offwind-ac', 'solar').
        Each dataset must contain variables 'potential' (MW) and 'profile' (0-1 CF timeseries).
    onshore_regions_gpd : gpd.GeoDataFrame
        Onshore regions with ISO3 country codes in 'country' column.
    offshore_regions_gpd : gpd.GeoDataFrame
        Offshore regions with 'OFF_' prefixed names to avoid ID collisions.
    country_codes : list[str], optional
        Filter to specific countries (ISO2 or ISO3). If None, uses all regions.
    total_clusters : int, default=10
        Target number of clusters (split by onshore_ratio if separate_onshore_offshore=True).
    onshore_ratio : float, default=0.8
        Fraction of total_clusters allocated to onshore (remainder to offshore).
    random_state : int, default=42
        Seed for KMeans reproducibility.
    separate_onshore_offshore : bool, default=True
        If True, cluster onshore and offshore separately within each country.
    
    Returns
    -------
    clusters_ds : xr.Dataset
        Clustered dataset with dimensions [cluster, technology, hour].
        
        Data variables:
          - renewable_potential: Sum of raw potentials per cluster-technology (MW).
                                 Preserved exactly during clustering - no energy loss.
          - avg_cf: Mean capacity factor from clustered regions (0-1).
                    Averaged timeseries introduces distortion—may not represent 
                    individual region characteristics.
          - p_nom_max: Installed capacity needed to preserve renewable_potential
                       with clustered CF timeseries: = renewable_potential / avg_cf (MW).
                       Compare to raw region p_nom_max to quantify clustering distortion.
          - capacity_factor: Hourly profiles [cluster, technology, hour].
                            Average of regional CF timeseries.
        
        Coordinates:
          - cluster: Unique cluster IDs (format: ISO3_[ON|OFF]_##)
          - iso3: Country code per cluster
          - technology: Available technologies
          - hour: Hour of year (0-8759)
          - lat, lon: Cluster centroid coordinates
          - area_km2: Total area of regions in cluster
    """

    # Calculate cluster counts based on ratio
    n_clusters_onshore = max(1, round(total_clusters * onshore_ratio))
    n_clusters_offshore = max(1, round(total_clusters * (1 - onshore_ratio)))
    
    # Safety check: verify the sum matches user-specified total
    total_allocated = n_clusters_onshore + n_clusters_offshore
    if total_allocated == total_clusters:
        logging.info(f"✓ Cluster allocation valid: {n_clusters_onshore} onshore + {n_clusters_offshore} offshore = {total_clusters} total")
    else:
        logging.warning(
            f"⚠ Cluster allocation mismatch: {n_clusters_onshore} onshore + {n_clusters_offshore} offshore = {total_allocated}, "
            f"but user requested {total_clusters} total. Difference: {total_allocated - total_clusters} clusters."
        )
    
    # Prepare regions (already have ISO3 country codes)
    all_regions = pd.concat([
        onshore_regions_gpd.assign(type='onshore'),
        offshore_regions_gpd.assign(type='offshore')
    ], ignore_index=True)
    
    all_regions['iso3'] = all_regions['country']
    # Extract geographic coordinates for grid matching (warning suppressed - intentional use of geographic CRS)
    import warnings
    with warnings.catch_warnings():
        warnings.filterwarnings('ignore', message='Geometry is in a geographic CRS')
        all_regions['lon'] = all_regions.geometry.centroid.x
        all_regions['lat'] = all_regions.geometry.centroid.y
    # Also compute projected centroid for KMeans (more accurate for clustering)
    all_regions_projected = all_regions.to_crs('EPSG:3857')
    all_regions['x_proj'] = all_regions_projected.geometry.centroid.x
    all_regions['y_proj'] = all_regions_projected.geometry.centroid.y
    
    # Filter by country codes if specified
    if country_codes is not None:
        iso3_filter = []
        for code in country_codes:
            try:
                if len(code) == 2:
                    iso3_filter.append(pycountry.countries.get(alpha_2=code.upper()).alpha_3)
                else:
                    iso3_filter.append(code.upper())
            except AttributeError:
                logging.warning(f"Unknown country code: {code}")
        all_regions = all_regions[all_regions['iso3'].isin(iso3_filter)]
    
    technologies = list(tech_profiles_nc.keys())
    iso3_codes = sorted(all_regions['iso3'].unique())
    
    logging.info(f"Clustering {len(iso3_codes)} countries: {iso3_codes}")
    logging.info(f"Separate onshore/offshore: {separate_onshore_offshore}")
    
    cluster_list = []
    
    for iso3 in iso3_codes:
        iso3_regions = all_regions[all_regions['iso3'] == iso3]
        logging.info(f"\n{iso3}: {len(iso3_regions)} regions")
        
        # Cluster onshore and offshore separately if requested
        if separate_onshore_offshore:
            regions_to_cluster = [
                ('onshore', iso3_regions[iso3_regions['type'] == 'onshore'].copy()),
                ('offshore', iso3_regions[iso3_regions['type'] == 'offshore'].copy())
            ]
        else:
            regions_to_cluster = [('all', iso3_regions.copy())]
        
        # Cluster each type
        for region_type, type_regions in regions_to_cluster:
            if len(type_regions) == 0:
                logging.info(f"  Skipping {region_type}: no regions available")
                continue
            
            # Determine requested clusters
            if region_type == 'onshore':
                requested_clusters = n_clusters_onshore
            elif region_type == 'offshore':
                requested_clusters = n_clusters_offshore
            else:
                requested_clusters = total_clusters
            
            # Fallback if not enough regions
            n_clusters = min(requested_clusters, len(type_regions))
            if n_clusters < requested_clusters:
                logging.warning(
                    f"  {region_type}: Requested {requested_clusters} clusters but only "
                    f"{len(type_regions)} regions available. Creating {n_clusters} clusters instead."
                )
            else:
                logging.info(f"  Clustering {region_type}: {len(type_regions)} regions → {n_clusters} clusters")
            
            kmeans = KMeans(n_clusters=n_clusters, random_state=random_state)
            # Use projected coordinates for accurate geographic clustering
            region_coords = type_regions[['x_proj', 'y_proj']].values
            type_regions['cluster_id'] = kmeans.fit_predict(region_coords)
            
            # Aggregate all technologies within each cluster
            for cluster_id in range(n_clusters):
                cluster_regions = type_regions[type_regions['cluster_id'] == cluster_id]
                region_names = cluster_regions['name'].values
                
                # Generate cluster ID
                if separate_onshore_offshore and region_type != 'all':
                    type_label = 'ON' if region_type == 'onshore' else 'OFF'
                    cluster_id_str = f"{iso3}_{type_label}_{cluster_id+1:02d}"
                else:
                    cluster_id_str = f"{iso3}_{cluster_id+1:02d}"
                
                # Spatial aggregation
                lat = cluster_regions['lat'].mean()
                lon = cluster_regions['lon'].mean()
                area_km2 = cluster_regions['area_km2'].sum()
                
                # Aggregate each technology
                for tech in technologies:
                    tech_ds = tech_profiles_nc[tech]
                    
                    # Get valid regions (strip OFF_ prefix for bus matching)
                    valid_regions = []
                    for r in region_names:
                        bus_name = r.replace('OFF_', '') if r.startswith('OFF_') else r
                        if str(bus_name) in [str(b) for b in tech_ds.bus.values]:
                            valid_regions.append(r)
                    
                    if len(valid_regions) == 0:
                        logging.warning(f"    {tech}: No regions in cluster {cluster_id_str}")
                        renewable_potential, avg_cf, cf_array = 0, 0, np.zeros(8760)
                    else:
                        # Average timeseries across all regions in cluster (preserves energy conservation)
                        cf_lists = []
                        for r in valid_regions:
                            bus_name = r.replace('OFF_', '') if r.startswith('OFF_') else r
                            cf_lists.append(tech_ds.sel(bus=bus_name).profile.values)
                        # Hourly capacity factor: averaged across all regions in this cluster
                        cf_array = np.mean(cf_lists, axis=0)
                        # Annual mean capacity factor for this cluster-technology combination
                        avg_cf = cf_array.mean()
                        
                        # Sum renewable generation potential across all regions in cluster (MW)
                        # This exact sum is preserved through clustering - no energy loss
                        region_rows = cluster_regions[cluster_regions['name'].isin(valid_regions)]
                        potential_list = []
                        for _, region in region_rows.iterrows():
                            lon_val, lat_val = region['lon'], region['lat']
                            x_idx = np.argmin(np.abs(tech_ds.x.values - lon_val))
                            y_idx = np.argmin(np.abs(tech_ds.y.values - lat_val))
                            potential_val = tech_ds['potential'].values[y_idx, x_idx]
                            potential_list.append(potential_val)
                        # Total renewable generation potential (MW) for cluster-technology
                        renewable_potential = np.sum(potential_list)
                    
                    # Store cluster aggregations for later xarray construction
                    cluster_list.append({
                        'cluster_id': cluster_id_str,
                        'iso3': iso3,
                        'type': region_type if separate_onshore_offshore else 'mixed',
                        'technology': tech,
                        'lat': lat,
                        'lon': lon,
                        'area_km2': area_km2,
                        'renewable_potential': renewable_potential,  # Sum of raw potentials (MW)
                        'avg_cf': avg_cf,  # Annual mean capacity factor (0-1)
                        'cf_timeseries': cf_array,  # Hourly profiles for 8760 hours
                    })
    
    # Build xarray dataset with all cluster aggregations and time series
    cluster_meta = pd.DataFrame(cluster_list)
    
    # Pre-compute data arrays for xarray construction
    renewable_potential_array = cluster_meta.pivot_table(
        index='cluster_id', columns='technology',
        values='renewable_potential', aggfunc='first'
    ).reindex(technologies, axis=1).values
    
    avg_cf_array = cluster_meta.pivot_table(
        index='cluster_id', columns='technology',
        values='avg_cf', aggfunc='first'
    ).reindex(technologies, axis=1).values
    
    # Compute p_nom_max as backward derivation: renewable_potential / avg_cf
    # Safe division handles division by zero (where avg_cf == 0, result is 0)
    p_nom_max_array = np.divide(
        renewable_potential_array,
        avg_cf_array,
        where=avg_cf_array > 0,
        out=np.zeros_like(avg_cf_array)
    )
    
    clusters_ds = xr.Dataset(
        data_vars={
            # Renewable generation potential per cluster-technology (MW) - exact sum, no distortion
            'renewable_potential': (['cluster', 'technology'], renewable_potential_array),
            # Annual mean capacity factor (0-1) - averaged timeseries introduces clustering distortion
            'avg_cf': (['cluster', 'technology'], avg_cf_array),
            # Hourly capacity factor profiles (0-1) - to be used in timeseries optimization
            'capacity_factor': (['cluster', 'technology', 'hour'],
                               np.array([c['cf_timeseries'] for c in cluster_list])
                                   .reshape(len(cluster_list)//len(technologies), len(technologies), 8760)),
            # Installed capacity needed to preserve renewable_potential with clustered CF timeseries: = renewable_potential / avg_cf (MW).
            'p_nom_max': (['cluster', 'technology'], p_nom_max_array),
        },
        coords={
            'cluster': sorted(cluster_meta['cluster_id'].unique()),  # Unique cluster identifiers
            # Country code for each cluster (for filtering/grouping by country)
            'iso3': (['cluster'], [cluster_meta[cluster_meta['cluster_id']==c]['iso3'].iloc[0]
                                  for c in sorted(cluster_meta['cluster_id'].unique())]),
            'technology': technologies,  # Technologies included (onwind, offwind-ac, solar)
            'hour': np.arange(8760),  # Hours in a year (0-8759)
            # Geographic centroid coordinates for cluster visualization and spatial reference
            'lat': (['cluster'], [cluster_meta[cluster_meta['cluster_id']==c]['lat'].iloc[0]
                                 for c in sorted(cluster_meta['cluster_id'].unique())]),
            'lon': (['cluster'], [cluster_meta[cluster_meta['cluster_id']==c]['lon'].iloc[0]
                                 for c in sorted(cluster_meta['cluster_id'].unique())]),
            # Total area of regions included in each cluster (km²) - for density/intensity calculations
            'area_km2': (['cluster'], [cluster_meta[cluster_meta['cluster_id']==c]['area_km2'].iloc[0]
                                      for c in sorted(cluster_meta['cluster_id'].unique())]),
        }
    )
    
    logging.info(f"\n✓ Created {len(clusters_ds.cluster)} geographic clusters × {len(technologies)} techs")
    return clusters_ds, cluster_meta


In [7]:
clusters_xr, cluster_metadata = cluster_renewable_potentials(
    tech_profiles_nc,
    onshore_regions_gpd,
    offshore_regions_gpd,
    total_clusters=10,
    onshore_ratio=0.8,
    separate_onshore_offshore=True
)

..\..\AppData\Local\Temp\ipykernel_10284\3139860479.py - ✓ Cluster allocation valid: 8 onshore + 2 offshore = 10 total
..\..\AppData\Local\Temp\ipykernel_10284\3139860479.py - Clustering 3 countries: ['IRL', 'NLD', 'PRT']
..\..\AppData\Local\Temp\ipykernel_10284\3139860479.py - Separate onshore/offshore: True
..\..\AppData\Local\Temp\ipykernel_10284\3139860479.py - 
IRL: 296 regions
..\..\AppData\Local\Temp\ipykernel_10284\3139860479.py -   Clustering onshore: 228 regions → 8 clusters
..\..\AppData\Local\Temp\ipykernel_10284\3139860479.py -   Clustering offshore: 68 regions → 2 clusters
..\..\AppData\Local\Temp\ipykernel_10284\3139860479.py - 
NLD: 346 regions
..\..\AppData\Local\Temp\ipykernel_10284\3139860479.py -   Clustering onshore: 291 regions → 8 clusters
..\..\AppData\Local\Temp\ipykernel_10284\3139860479.py -     offwind-ac: No regions in cluster NLD_ON_01
..\..\AppData\Local\Temp\ipykernel_10284\3139860479.py -     offwind-ac: No regions in cluster NLD_ON_02
..\..\AppData\Loc

In [8]:
def validate_clustering(cluster_metadata, clusters_xr, tech_profiles_nc, onshore_regions_gpd, offshore_regions_gpd):
    """Validate: Compare raw vs clustered potentials (ISO3 codes only)."""
    import pandas as pd
    import numpy as np
    
    # Raw potentials - sum from source regions before clustering
    all_regions = pd.concat([
        onshore_regions_gpd.assign(type='onshore'),
        offshore_regions_gpd.assign(type='offshore')
    ], ignore_index=True)
    
    techs = list(tech_profiles_nc.keys())
    
    raw_data = []
    for _, r in all_regions.iterrows():
        lon, lat = r.geometry.centroid.x, r.geometry.centroid.y
        bus = r['name'].replace('OFF_', '') if r['type'] == 'offshore' else r['name']
        
        for tech in techs:
            ds = tech_profiles_nc[tech]
            if str(bus) not in [str(b) for b in ds.bus.values]:
                continue
            x_idx = np.argmin(np.abs(ds.x.values - lon))
            y_idx = np.argmin(np.abs(ds.y.values - lat))
            raw_data.append({'iso3': r['country'], 'tech': tech, 'value': ds['potential'].values[y_idx, x_idx]})
    
    # Raw totals per country-technology (should be preserved exactly)
    raw_tots = pd.DataFrame(raw_data).groupby(['iso3', 'tech'])['value'].sum()
    
    # Clustered potentials totals per country-technology (renewable_potential = exact sum, no loss)
    cluster_tots = cluster_metadata.groupby(['iso3', 'technology'])['renewable_potential'].sum()
    
    # Compare - should be identical (no energy loss during clustering)
    all_keys = set(raw_tots.index) | set(cluster_tots.index)
    comp_data = []
    for iso3, tech in sorted(all_keys):
        r_val, c_val = raw_tots.get((iso3, tech), 0), cluster_tots.get((iso3, tech), 0)
        comp_data.append({'iso3': iso3, 'tech': tech, 'raw': r_val, 'clustered': c_val, 
                         'diff': c_val - r_val, 'match': '✓' if abs(c_val - r_val) < 0.01 else '✗'})
    comp = pd.DataFrame(comp_data) if comp_data else pd.DataFrame()
    
    # Report validation results
    all_match = (comp['diff'].abs() < 0.01).all() if len(comp) > 0 else True
    max_diff = comp['diff'].abs().max() if len(comp) > 0 else 0
    matches = (comp['match'] == '✓').sum() if len(comp) > 0 else 0
    print(f"\n[VALIDATION] {len(comp)} entries | {matches} perfect | Max diff: {max_diff:.4f} GW | {'✓ PASS' if all_match else '✗ FAIL'}\n")
    
    return {'raw': raw_tots, 'clustered': cluster_tots, 'comparison': comp, 'valid': all_match}


In [9]:
# Validate: Compare raw vs clustered potentials
validation_results = validate_clustering(
    cluster_metadata,
    clusters_xr,
    tech_profiles_nc,
    onshore_regions_gpd,
    offshore_regions_gpd
)



[VALIDATION] 9 entries | 9 perfect | Max diff: 0.0000 GW | ✓ PASS



In [10]:
def save_clusters(clusters_xr, cluster_metadata, output_dir="data", filename_prefix="renewable_clusters"):
    """
    Save clustered renewable potential data to NetCDF and metadata to CSV.
    
    Parameters:
    -----------
    clusters_xr : xr.Dataset
        Clustered potentials xarray dataset
    cluster_metadata : pd.DataFrame
        Cluster metadata DataFrame
    output_dir : str
        Output directory (created if doesn't exist)
    filename_prefix : str
        Prefix for output files
    
    Returns:
    --------
    paths : dict
        Dictionary with keys 'xarray' and 'metadata' pointing to saved files
    """
    import os
    from datetime import datetime
    
    # Create output directory if needed
    os.makedirs(output_dir, exist_ok=True)
    
    # Generate timestamp-based filenames
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    xr_path = os.path.join(output_dir, f"{filename_prefix}_{timestamp}.nc")
    meta_path = os.path.join(output_dir, f"{filename_prefix}_metadata_{timestamp}.csv")
    
    # Save xarray to NetCDF
    clusters_xr.to_netcdf(xr_path)
    logging.info(f"✓ Saved clustered potentials: {xr_path}")
    
    # Save metadata to CSV
    cluster_metadata.to_csv(meta_path, index=False)
    logging.info(f"✓ Saved cluster metadata: {meta_path}")
    
    return {'xarray': xr_path, 'metadata': meta_path}

# Save the results
save_paths = save_clusters(clusters_xr, cluster_metadata, output_dir="data")


..\..\AppData\Local\Temp\ipykernel_10284\1196717265.py - ✓ Saved clustered potentials: data\renewable_clusters_20260325_172014.nc
..\..\AppData\Local\Temp\ipykernel_10284\1196717265.py - ✓ Saved cluster metadata: data\renewable_clusters_metadata_20260325_172014.csv
